In [26]:
%matplotlib

import torch
import numpy as np
import matplotlib.pyplot as plt

Using matplotlib backend: TkAgg


In [27]:
trunk_x_0 = torch.tensor([[0, 0, 0.3]])
trunk_xd_0 = torch.zeros_like(trunk_x_0)
trunk_o_0 = torch.zeros_like(trunk_x_0)
trunk_od_0 = torch.zeros_like(trunk_x_0)

In [28]:
trunk_x_lo = torch.tensor([[0.0881, 0.0000, 0.2838]])
trunk_xd_lo = torch.tensor([[1.4436, 0.0000, 1.1814]])
trunk_o_lo = torch.tensor([[-0.0017, -0.2083,  0.0152]])
trunk_od_lo = torch.tensor([[-0.1030,  2.5308, -0.1150]])
t_th = torch.tensor([[0.4067]])

In [29]:
def BezierTrajectory(start: torch.Tensor, start_v: torch.Tensor, end: torch.Tensor, end_v: torch.Tensor, t_th: torch.Tensor, t_ex: float):

    w = torch.stack((
        start.unsqueeze(1),
        (((t_th / 3) * start_v) + start).unsqueeze(1),
        ((-(t_th / 3) * end_v) + end).unsqueeze(1),
        end.unsqueeze(1)), dim=2).squeeze()

    w = w.reshape(-1, 4, 3)

    w_d = torch.stack((
        ((3 / t_th) * (w[:, 1] - w[:, 0])).unsqueeze(1),
        ((3 / t_th) * (w[:, 2] - w[:, 1])).unsqueeze(1),
        ((3 / t_th) * (w[:, 3] - w[:, 2])).unsqueeze(1)), dim=2).squeeze()

    t = t_ex / t_th

    bezier_curve_3 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(3 - 0),
        (3.0) * (t**1) * (1 - t)**(3 - 1),
        (3.0) * (t**2) * (1 - t)**(3 - 2),
        (1.0) * (t**3) * (1 - t)**(3 - 3),
    ), dim=-1)

    bezier_curve_2 = torch.cat((
        (1.0) * (t**0) * (1 - t)**(2 - 0),
        (2.0) * (t**1) * (1 - t)**(2 - 1),
        (1.0) * (t**2) * (1 - t)**(2 - 2),
    ), dim=-1)

    bezier_position = torch.cat((
        (w[..., 0] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 1] * bezier_curve_3).sum(dim=-1).reshape(-1, 1),
        (w[..., 2] * bezier_curve_3).sum(dim=-1).reshape(-1, 1)), dim=1)

    bezier_velocity = torch.cat((
        (w_d[..., 0] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 1] * bezier_curve_2).sum(dim=-1).reshape(-1, 1),
        (w_d[..., 2] * bezier_curve_2).sum(dim=-1).reshape(-1, 1)), dim=1)

    return bezier_position, bezier_velocity

In [32]:
max_time = t_th.item() + 0.005

In [33]:
pos_lin_vel = [BezierTrajectory(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
pos = torch.stack([val[0] for val in pos_lin_vel], dim=1)
lin_vel = torch.stack([val[1] for val in pos_lin_vel], dim=1)

orient_ang_vel = [BezierTrajectory(trunk_o_0, trunk_od_0, trunk_o_lo, trunk_od_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
orient = torch.stack([val[0] for val in orient_ang_vel], dim=1)
ang_vel = torch.stack([val[1] for val in orient_ang_vel], dim=1)

In [34]:
pos.shape, lin_vel.shape

(torch.Size([1, 83, 3]), torch.Size([1, 83, 3]))

In [35]:
orient.shape, ang_vel.shape

(torch.Size([1, 83, 3]), torch.Size([1, 83, 3]))

In [36]:
# fig = plt.figure(figsize=(10, 10))
# ax = plt.axes(projection='3d')
# ax.set_zlim(0, 0.35)
# ax.set_xlim(-0.35, 0.35)
# ax.set_ylim(-0.35, 0.35)
# ax.scatter(pos[..., 0], pos[..., 1], pos[..., 2], color='blue')
# ax.scatter(trunk_x_0[:, 0], trunk_x_0[:, 1], trunk_x_0[:, 2], alpha=1, linewidths=5)
# ax.scatter(trunk_x_lo[:, 0], trunk_x_lo[:, 1], trunk_x_lo[:, 2], alpha=1, linewidths=5)

# plt.show()

In [37]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
step_size = 5
draw_l_vel = False
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
if draw_l_vel:
    ax.quiver(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2],
              lin_vel[i][..., 0], lin_vel[i][..., 1], lin_vel[i][..., 2],
              length=0.1, color='purple', arrow_length_ratio=0)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

for j in range(0, len(orient[i]), step_size):
    # Getting the current position and orientation
    p = pos[i, j].numpy()
    o = orient[i, j].numpy()

    # Define the Cartesian frame vectors (x, y, z)
    x_vec = np.array([1, 0, 0])  # X-axis
    y_vec = np.array([0, 1, 0])  # Y-axis
    z_vec = np.array([0, 0, 1])  # Z-axis

    # Rotate the frame based on orientation
    rotation_matrix = np.eye(3)
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [1, 0, 0],
        [0, np.cos(o[0]), -np.sin(o[0])],
        [0, np.sin(o[0]), np.cos(o[0])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[1]), 0, np.sin(o[1])],
        [0, 1, 0],
        [-np.sin(o[1]), 0, np.cos(o[1])]
    ]))
    rotation_matrix = np.matmul(rotation_matrix, np.array([
        [np.cos(o[2]), -np.sin(o[2]), 0],
        [np.sin(o[2]), np.cos(o[2]), 0],
        [0, 0, 1]
    ]))

    x_vec = np.matmul(rotation_matrix, x_vec)
    y_vec = np.matmul(rotation_matrix, y_vec)
    z_vec = np.matmul(rotation_matrix, z_vec)

    # Plotting the frame
    ax.quiver(p[0], p[1], p[2], x_vec[0], x_vec[1], x_vec[2], length=0.05, color='r', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], y_vec[0], y_vec[1], y_vec[2], length=0.05, color='g', arrow_length_ratio=0)
    ax.quiver(p[0], p[1], p[2], z_vec[0], z_vec[1], z_vec[2], length=0.05, color='b', arrow_length_ratio=0)

ax.quiver(pos[...,-1, 0], pos[...,-1, 1], pos[...,-1, 2], lin_vel[...,-1, 0], lin_vel[...,-1, 1], lin_vel[...,-1, 2], length=0.1, normalize=False, color='purple', arrow_length_ratio=0.1)

plt.show()

Adding velocity

In [44]:
v0 = trunk_xd_lo
v_mult = 1.0319
vf = v0 * v_mult

In [45]:
vf

tensor([[1.4897, 0.0000, 1.2191]])

In [46]:
v_un = vf / np.linalg.norm(vf)

In [47]:
s0 = trunk_x_lo

In [48]:
l_exp = 0.0921

In [49]:
sf = s0 + v_un * l_exp

In [50]:
sf

tensor([[0.1594, 0.0000, 0.3421]])

In [51]:
vf_n = np.linalg.norm(vf)
v0_n = np.linalg.norm(v0)
sf_n = np.linalg.norm(sf)
s0_n = np.linalg.norm(s0)

In [52]:
a = 0.5 * ((np.power(vf_n, 2)-np.power(v0_n, 2))/(sf_n-s0_n))

In [53]:
a, a/9.81

(1.4049306784744138, 0.14321413643979752)

In [54]:
t_exp = (vf_n-v0_n)/a

In [55]:
t_exp

0.04235515645337645

In [56]:
t_th+ t_exp

tensor([[0.4491]])

In [57]:
sim_time = np.arange(0,t_exp+0.005,0.005)

In [58]:
sim_time

array([0.   , 0.005, 0.01 , 0.015, 0.02 , 0.025, 0.03 , 0.035, 0.04 ,
       0.045])

In [59]:
s = torch.stack([torch.lerp(torch.tensor(s0), torch.tensor(sf), t/t_exp) for t in sim_time])
s

/tmp/ipykernel_934882/2118805912.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  s = torch.stack([torch.lerp(torch.tensor(s0), torch.tensor(sf), t/t_exp) for t in sim_time])


tensor([[[0.0881, 0.0000, 0.2838]],

        [[0.0965, 0.0000, 0.2907]],

        [[0.1049, 0.0000, 0.2976]],

        [[0.1133, 0.0000, 0.3045]],

        [[0.1218, 0.0000, 0.3113]],

        [[0.1302, 0.0000, 0.3182]],

        [[0.1386, 0.0000, 0.3251]],

        [[0.1470, 0.0000, 0.3320]],

        [[0.1554, 0.0000, 0.3389]],

        [[0.1638, 0.0000, 0.3458]]])

In [60]:
v = torch.stack([torch.lerp(torch.tensor(v0), torch.tensor(vf), t/t_exp) for t in sim_time if t <= t_exp])
v

/tmp/ipykernel_934882/99624924.py:1: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  v = torch.stack([torch.lerp(torch.tensor(v0), torch.tensor(vf), t/t_exp) for t in sim_time if t <= t_exp])


tensor([[[1.4436, 0.0000, 1.1814]],

        [[1.4490, 0.0000, 1.1858]],

        [[1.4545, 0.0000, 1.1903]],

        [[1.4599, 0.0000, 1.1947]],

        [[1.4653, 0.0000, 1.1992]],

        [[1.4708, 0.0000, 1.2036]],

        [[1.4762, 0.0000, 1.2081]],

        [[1.4817, 0.0000, 1.2125]],

        [[1.4871, 0.0000, 1.2170]]])

In [61]:
# plt.close()

#fig = plt.figure(figsize=(10, 10))
#ax = plt.axes(projection='3d')
#ax.set_zlim(0, 0.4)
#ax.set_xlim(-0.2, 0.2)
#ax.set_ylim(-0.2, 0.2)
#ax.set_xlabel("x")
#ax.set_ylabel("y")
#ax.set_zlabel("z")
#
#ax.scatter(s[..., 0], s[..., 1], s[..., 2], color='black', linewidths=1)
#ax.quiver(s[..., 0], s[..., 1], s[..., 2], v[...,0], v[...,1], v[...,2], length=0.1, normalize=False, color='purple', arrow_length_ratio=0.1)
#
#
#plt.show()

In [63]:
sf

tensor([[0.1594, 0.0000, 0.3421]])

In [79]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

ax.scatter(s[..., 0], s[..., 1], s[..., 2], color='black', linewidths=1)
ax.scatter(sf[..., 0], sf[..., 1], sf[..., 2], alpha=1, linewidths=5)
ax.quiver(pos[..., -1, 0], pos[..., -1, 1], pos[..., -1, 2], lin_vel[..., -1, 0], lin_vel[..., -1, 1], lin_vel[..., -1, 2], length=0.1, normalize=False, color='orange', arrow_length_ratio=0.1)
ax.quiver(s[..., 0][-1], s[..., 1][-1], s[..., 2][-1], v[..., 0][-1], v[..., 1][-1], v[..., 2][-1], length=0.1, normalize=False, color='green', arrow_length_ratio=0.1)


plt.show()

In [31]:
trunk_x_lo = torch.tensor([[0,0,0.35]])
trunk_xd_lo = torch.tensor([[0,0,2.5]])
t_th = torch.tensor([[0.5500]])

In [32]:
max_time = 0.5500 + 0.005

In [33]:
pos_lin_vel = [BezierTrajectory(trunk_x_0, trunk_xd_0, trunk_x_lo, trunk_xd_lo, t_th, t) for t in torch.arange(0, max_time, 0.005)]
pos = torch.stack([val[0] for val in pos_lin_vel], dim=1)
lin_vel = torch.stack([val[1] for val in pos_lin_vel], dim=1)

In [34]:
# plt.close()

fig = plt.figure(figsize=(10, 10))
ax = plt.axes(projection='3d')
ax.set_zlim(0, 0.4)
ax.set_xlim(-0.2, 0.2)
ax.set_ylim(-0.2, 0.2)
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("z")

i = 0
ax.scatter(pos[i][..., 0], pos[i][..., 1], pos[i][..., 2], color='black', linewidths=1)
ax.scatter(trunk_x_0[i, 0], trunk_x_0[i, 1], trunk_x_0[i, 2], alpha=1, linewidths=5)
ax.scatter(trunk_x_lo[i, 0], trunk_x_lo[i, 1], trunk_x_lo[i, 2], alpha=1, linewidths=5)

ax.quiver(pos[...,-1, 0], pos[...,-1, 1], pos[...,-1, 2], lin_vel[...,-1, 0], lin_vel[...,-1, 1], lin_vel[...,-1, 2], length=0.1, normalize=False, color='orange', arrow_length_ratio=0.1)


plt.show()